In [1]:
import warnings
warnings.filterwarnings('ignore')
import os
import pandas as pd
import scipy.stats as stats
import sklearn.metrics
import itertools

In [2]:
def load_performance_metrics_for_T_tests(approach, dataset, target_metric, trial):
  
  performance = list()

  random_states = [x for x in os.listdir(f'../../outputs/{approach.replace("-", "_").lower()}/{dataset}-{approach}/{trial}') if os.path.isdir(f'../../outputs/{approach.replace("-", "_").lower()}/{dataset}-{approach}/{trial}/{x}')]
  if len(random_states) < 10:
    print('Not enough random states...')

  for random_state in random_states:
    df = pd.read_csv(f'../../outputs/{approach.replace("-", "_").lower()}/{dataset}-{approach}/{trial}/{random_state}/predictions.csv')
    for split in ['validation', 'test']:
      df_split = df[df['split'] == split]
      accuracy = sklearn.metrics.accuracy_score(df_split['real'], df_split['prediction'])
      f1_score = sklearn.metrics.f1_score(df_split['real'], df_split['prediction'], average = 'macro')
      precision = sklearn.metrics.precision_score(df_split['real'], df_split['prediction'], average = 'macro')
      recall = sklearn.metrics.recall_score(df_split['real'], df_split['prediction'], average = 'macro')
      performance.append((trial, random_state, split, accuracy, f1_score, precision, recall))  
  return pd.DataFrame(performance, columns = ['trial', 'random_state', 'split', 'accuracy', 'f1_score', 'precision', 'recall'])[['random_state', 'split', target_metric]].rename(columns = {target_metric : 'performance'})

In [3]:
def get_all_performance_metrics_for_T_tests(approach, datasets, target_metrics, trials):
  l = list()
  for dataset, target_metric, trial in zip(datasets, target_metrics, trials):
    l.append(
      load_performance_metrics_for_T_tests(
        approach = approach,
        dataset = dataset,
        trial = trial,
        target_metric = target_metric,
      ).assign(
        approach = approach,
        dataset = dataset,
      )
    )
  return pd.concat(l, axis = 0).reset_index(drop = True)

# Core Experiments

In [4]:
standard_vs_truncated = pd.concat([
  get_all_performance_metrics_for_T_tests(
    approach = 'Fine-tuning-Truncated',
    datasets = ['SST-2', 'Ohsumed', 'R8', 'IMDb-1k'],
    target_metrics = ['accuracy', 'f1_score', 'f1_score', 'accuracy'],
    trials = [0, 78, 71, 65]
  ),
  get_all_performance_metrics_for_T_tests(
    approach = 'Fine-tuning',
    datasets = ['SST-2', 'Ohsumed', 'R8', 'IMDb-1k'],
    target_metrics = ['accuracy', 'f1_score', 'f1_score', 'accuracy'],
    trials = [104, 0, 0, 56]
  ),
  get_all_performance_metrics_for_T_tests(
    approach = 'Sliding_windows',
    datasets = ['SST-2', 'Ohsumed', 'R8', 'IMDb-1k'],
    target_metrics = ['accuracy', 'f1_score', 'f1_score', 'accuracy'],
    trials = [0, 44, 100, 9]
  ),
  get_all_performance_metrics_for_T_tests(
    approach = 'Attention_distillation',
    datasets = ['SST-2', 'Ohsumed', 'R8', 'IMDb-1k'],
    target_metrics = ['accuracy', 'f1_score', 'f1_score', 'accuracy'],
    trials = [0, 73, 69, 104]
  ),
  get_all_performance_metrics_for_T_tests(
    approach = 'Chefer_Importance',
    datasets = ['SST-2', 'Ohsumed', 'R8', 'IMDb-1k'],
    target_metrics = ['accuracy', 'f1_score', 'f1_score', 'accuracy'],
    trials = [74, 55, 38, 0]
  )
])

In [5]:
for dataset in ['SST-2', 'Ohsumed', 'R8', 'IMDb-1k']:
  print('-' * 10, dataset, '-' * 10)
  for i, split in enumerate(['validation', 'test']):
    print('>>', split)
    split_df = standard_vs_truncated[(standard_vs_truncated['dataset'] == dataset) & (standard_vs_truncated['split'] == split)].drop(columns = ['split']) \
      .pivot(index = ['dataset', 'random_state'], columns = 'approach', values = 'performance') \
      .reset_index() \
      .sort_values(by = ['dataset', 'random_state'])
    
    if dataset == 'IMDb':
      combinations = ['Fine-tuning', 'Fine-tuning-Truncated']
    else:
      combinations = ['Fine-tuning', 'Fine-tuning-Truncated', 'Sliding_windows', 'Attention_distillation', 'Chefer_Importance']
    for (x, y) in itertools.combinations(combinations, 2):
      # H0: Mean Grouped and Surrogate scores are equal
      # H1: Mean Grouped and Surrogate scores are not equal
      statistic, p_value = stats.ttest_ind(split_df[x], split_df[y])
      # 95% confidence -- p-value < 0.05 => reject the null hypothesis (H0), i.e., the true mean test score is different between the approaches
      print(f'{x} - {y}:', 'Statistic:', statistic, 'P-value:', p_value, 'Reject H0?', p_value < 0.05)
  print('')

---------- SST-2 ----------
>> validation
Fine-tuning - Fine-tuning-Truncated: Statistic: -0.5318965725889424 P-value: 0.6013015272697095 Reject H0? False
Fine-tuning - Sliding_windows: Statistic: -0.8193530401979925 P-value: 0.4233016220568261 Reject H0? False
Fine-tuning - Attention_distillation: Statistic: 0.0 P-value: 1.0 Reject H0? False
Fine-tuning - Chefer_Importance: Statistic: -1.697066721581731 P-value: 0.10690728422666242 Reject H0? False
Fine-tuning-Truncated - Sliding_windows: Statistic: -0.22742941307362763 P-value: 0.8226530480821893 Reject H0? False
Fine-tuning-Truncated - Attention_distillation: Statistic: 0.7138380564460509 P-value: 0.48448144335622 Reject H0? False
Fine-tuning-Truncated - Chefer_Importance: Statistic: -1.3249788429921356 P-value: 0.20175314120416318 Reject H0? False
Sliding_windows - Attention_distillation: Statistic: 1.3627406161774862 P-value: 0.1897715556816901 Reject H0? False
Sliding_windows - Chefer_Importance: Statistic: -1.841727395407638 P-v

## Ablations

In [4]:
def load_ablation_performance_metrics_for_T_tests(approach, dataset, target_metric, trial, ablation):
  
  performance = list()

  random_states = [x for x in os.listdir(f'../../outputs/ablations/{dataset}-{approach}-{ablation}/{trial}') if os.path.isdir(f'../../outputs/ablations/{dataset}-{approach}-{ablation}/{trial}/{x}')]
  if len(random_states) < 10:
    print('Not enough random states...')

  for random_state in random_states:
    df = pd.read_csv(f'../../outputs/ablations/{dataset}-{approach}-{ablation}/{trial}/{random_state}/predictions.csv')
    for split in ['validation', 'test']:
      df_split = df[df['split'] == split]
      accuracy = sklearn.metrics.accuracy_score(df_split['real'], df_split['prediction'])
      f1_score = sklearn.metrics.f1_score(df_split['real'], df_split['prediction'], average = 'macro')
      precision = sklearn.metrics.precision_score(df_split['real'], df_split['prediction'], average = 'macro')
      recall = sklearn.metrics.recall_score(df_split['real'], df_split['prediction'], average = 'macro')
      performance.append((trial, random_state, split, accuracy, f1_score, precision, recall))  
  return pd.DataFrame(performance, columns = ['trial', 'random_state', 'split', 'accuracy', 'f1_score', 'precision', 'recall'])[['random_state', 'split', target_metric]].rename(columns = {target_metric : 'performance'})

In [5]:
def get_all_ablation_performance_metrics_for_T_tests(approach, datasets, target_metrics, trials, ablations):
  l = list()
  for dataset, target_metric, trial in zip(datasets, target_metrics, trials):
    for ablation in ablations:
      l.append(
        load_ablation_performance_metrics_for_T_tests(
          approach = approach,
          dataset = dataset,
          trial = trial,
          target_metric = target_metric,
          ablation = ablation,
        ).assign(
          approach = approach,
          dataset = dataset,
          ablation = ablation,
        )
      )
  return pd.concat(l, axis = 0).reset_index(drop = True)

In [6]:
standard_vs_truncated = pd.concat([
  get_all_performance_metrics_for_T_tests(
    approach = 'Chefer_Importance',
    datasets = ['SST-2', 'Ohsumed', 'R8', 'IMDb-1k'],
    target_metrics = ['accuracy', 'f1_score', 'f1_score', 'accuracy'],
    trials = [74, 55, 38, 0]
  ).assign(ablation = 'None'),
  get_all_ablation_performance_metrics_for_T_tests(
    approach = 'Chefer_Importance',
    datasets = ['SST-2', 'Ohsumed', 'R8', 'IMDb-1k'],
    target_metrics = ['accuracy', 'f1_score', 'f1_score', 'accuracy'],
    trials = [74, 55, 38, 0],
    ablations = ['fully_connected_first_level_edges', 'drop_first_level_edges', 'unit_weight_edges', 'drop_second_level_nodes', 'drop_second_and_third_level_nodes', 'bi_directional_edges_to_second_and_third_level_nodes', 'no_node_pruning', 'no_node_and_edge_pruning']
  )
])

In [9]:
for dataset in ['SST-2', 'Ohsumed', 'R8', 'IMDb-1k']:
  print('-' * 10, dataset, '-' * 10)
  for i, split in enumerate(['validation', 'test']):
    print('>>', split)
    split_df = standard_vs_truncated[(standard_vs_truncated['dataset'] == dataset) & (standard_vs_truncated['split'] == split)].drop(columns = ['split', 'approach']) \
      .pivot(index = ['dataset', 'random_state'], columns = 'ablation', values = 'performance') \
      .reset_index() \
      .sort_values(by = ['dataset', 'random_state'])
    
    combinations = ['None', 'fully_connected_first_level_edges', 'drop_first_level_edges', 'unit_weight_edges', 'drop_second_level_nodes', 'drop_second_and_third_level_nodes', 'bi_directional_edges_to_second_and_third_level_nodes', 'no_node_pruning', 'no_node_and_edge_pruning']
    for (x, y) in itertools.combinations(combinations, 2):
      # H0: Mean Grouped and Surrogate scores are equal
      # H1: Mean Grouped and Surrogate scores are not equal
      statistic, p_value = stats.ttest_ind(split_df[x], split_df[y])
      # 95% confidence -- p-value < 0.05 => reject the null hypothesis (H0), i.e., the true mean test score is different between the approaches
      print(f'{x} - {y}:', 'Statistic:', statistic, 'P-value:', p_value, 'Reject H0?', p_value < 0.05)
  print('')

---------- SST-2 ----------
>> validation
None - fully_connected_first_level_edges: Statistic: 0.29457100786286905 P-value: 0.7716918863734443 Reject H0? False
None - drop_first_level_edges: Statistic: -0.059467182344870556 P-value: 0.9532352583035445 Reject H0? False
None - unit_weight_edges: Statistic: -0.06275930889835453 P-value: 0.9506498351976882 Reject H0? False
None - drop_second_level_nodes: Statistic: 0.7202940575984653 P-value: 0.4805939473809585 Reject H0? False
None - drop_second_and_third_level_nodes: Statistic: -0.7521398046337235 P-value: 0.46168996237334425 Reject H0? False
None - bi_directional_edges_to_second_and_third_level_nodes: Statistic: 2.138241369896838 P-value: 0.04646312045511518 Reject H0? True
None - no_node_pruning: Statistic: 4.9640709100495135 P-value: 0.00010035512023064385 Reject H0? True
None - no_node_and_edge_pruning: Statistic: 5.50884456315946 P-value: 3.128823612377102e-05 Reject H0? True
fully_connected_first_level_edges - drop_first_level_edge

In [6]:
def load_downsample_ablation_performance_metrics_for_T_tests(approach, dataset, target_metric, trial, ablation):
  
  performance = list()

  random_states = [x for x in os.listdir(f'../../outputs/Chefer_importance_downsample_ablation/{dataset}-{approach}-{ablation}/{trial}') if os.path.isdir(f'../../outputs/Chefer_importance_downsample_ablation/{dataset}-{approach}-{ablation}/{trial}/{x}')]
  if len(random_states) < 10:
    print('Not enough random states...')

  for random_state in random_states:
    df = pd.read_csv(f'../../outputs/Chefer_importance_downsample_ablation/{dataset}-{approach}-{ablation}/{trial}/{random_state}/predictions.csv')
    for split in ['validation', 'test']:
      df_split = df[df['split'] == split]
      accuracy = sklearn.metrics.accuracy_score(df_split['real'], df_split['prediction'])
      f1_score = sklearn.metrics.f1_score(df_split['real'], df_split['prediction'], average = 'macro')
      precision = sklearn.metrics.precision_score(df_split['real'], df_split['prediction'], average = 'macro')
      recall = sklearn.metrics.recall_score(df_split['real'], df_split['prediction'], average = 'macro')
      performance.append((trial, random_state, split, accuracy, f1_score, precision, recall))  
  return pd.DataFrame(performance, columns = ['trial', 'random_state', 'split', 'accuracy', 'f1_score', 'precision', 'recall'])[['random_state', 'split', target_metric]].rename(columns = {target_metric : 'performance'})

In [7]:
def get_all_downsample_ablation_performance_metrics_for_T_tests(approach, datasets, target_metrics, trials, ablations):
  l = list()
  for dataset, target_metric, trial in zip(datasets, target_metrics, trials):
    for ablation in ablations:
      l.append(
        load_downsample_ablation_performance_metrics_for_T_tests(
          approach = approach,
          dataset = dataset,
          trial = trial,
          target_metric = target_metric,
          ablation = ablation,
        ).assign(
          approach = approach,
          dataset = dataset,
          ablation = ablation,
        )
      )
  return pd.concat(l, axis = 0).reset_index(drop = True)

In [8]:
training_data_downsample_ablation = pd.concat([
  get_all_performance_metrics_for_T_tests(
    approach = 'Chefer_Importance',
    datasets = ['SST-2', 'Ohsumed', 'R8', 'IMDb-1k'],
    target_metrics = ['accuracy', 'f1_score', 'f1_score', 'accuracy'],
    trials = [74, 55, 38, 0]
  ).assign(ablation = 'None'),
  get_all_downsample_ablation_performance_metrics_for_T_tests(
    approach = 'Chefer_Importance',
    datasets = ['SST-2', 'Ohsumed', 'R8', 'IMDb-1k'],
    target_metrics = ['accuracy', 'f1_score', 'f1_score', 'accuracy'],
    trials = [74, 55, 38, 0],
    ablations = ['0.25', '0.5', '0.75']
  )
])

In [9]:
for dataset in ['SST-2', 'Ohsumed', 'R8', 'IMDb-1k']:
  print('-' * 10, dataset, '-' * 10)
  for i, split in enumerate(['validation', 'test']):
    print('>>', split)
    split_df = training_data_downsample_ablation[(training_data_downsample_ablation['dataset'] == dataset) & (training_data_downsample_ablation['split'] == split)].drop(columns = ['split', 'approach']) \
      .pivot(index = ['dataset', 'random_state'], columns = 'ablation', values = 'performance') \
      .reset_index() \
      .sort_values(by = ['dataset', 'random_state'])
    
    combinations = ['None', '0.25', '0.5', '0.75']
    for (x, y) in itertools.combinations(combinations, 2):
      # H0: Mean Grouped and Surrogate scores are equal
      # H1: Mean Grouped and Surrogate scores are not equal
      statistic, p_value = stats.ttest_ind(split_df[x], split_df[y])
      # 95% confidence -- p-value < 0.05 => reject the null hypothesis (H0), i.e., the true mean test score is different between the approaches
      print(f'{x} - {y}:', 'Statistic:', statistic, 'P-value:', p_value, 'Reject H0?', p_value < 0.05)
  print('')

---------- SST-2 ----------
>> validation
None - 0.25: Statistic: 10.04712013421487 P-value: 8.307249340029321e-09 Reject H0? True
None - 0.5: Statistic: 13.952875194900283 P-value: 4.2988531064846834e-11 Reject H0? True
None - 0.75: Statistic: 10.88912903285918 P-value: 2.370758725533439e-09 Reject H0? True
0.25 - 0.5: Statistic: 1.066459945155672 P-value: 0.30031185890050793 Reject H0? False
0.25 - 0.75: Statistic: -1.037602468601144 P-value: 0.3131938392226967 Reject H0? False
0.5 - 0.75: Statistic: -2.611941734441812 P-value: 0.01765034825535498 Reject H0? True
>> test
None - 0.25: Statistic: 8.663584622227727 P-value: 7.736785941870801e-08 Reject H0? True
None - 0.5: Statistic: 10.726109716844766 P-value: 3.004932597554913e-09 Reject H0? True
None - 0.75: Statistic: -3.331885977408694 P-value: 0.003710570865589535 Reject H0? True
0.25 - 0.5: Statistic: -3.3450658169249627 P-value: 0.003604032408220723 Reject H0? True
0.25 - 0.75: Statistic: -10.266351253959067 P-value: 5.950495493